# Evaluation

In [ ]:
%load_ext autoreload
%autoreload 2
from typing import Literal

from pathlib import Path

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


sns.set_context("notebook", font_scale=0.1)
plt.style.use("default")
plt.rc("figure", dpi=300)
plt.rcParams["axes.titlesize"] = 10
plt.rcParams["axes.labelsize"] = 9
plt.rcParams["xtick.labelsize"] = 8
plt.rcParams["ytick.labelsize"] = 8
plt.rcParams["legend.fontsize"] = 8

from evaluate import PlottingWrapper

## Helpers

In [ ]:
plotting_wrapper = PlottingWrapper()

In [ ]:
input_dir = Path("data/evaluation/")

dfs = []
for file in input_dir.iterdir():
    dataset_name = file.stem
    if file.suffix != ".csv":
        continue
    raw_df = pd.read_csv(file, index_col=0)
    raw_df = raw_df.rename(columns={"iso_639_3": "category", "subject": "category"})

    raw_df.insert(0, "dataset", dataset_name)
    if "flores" in dataset_name or "alpaca" in dataset_name:
        raw_df = raw_df[raw_df["score"] != 1]
        raw_df.loc[raw_df["score"] == 2, "score"] -= 1

    dfs.append(raw_df)
df = pd.concat(dfs)
df: pd.DataFrame
df = df.reset_index()
length_base_df = df.copy()

In [ ]:
df["persona"] = df["persona"].str.replace("dynamic", "d")
df["persona"] = df["persona"].str.replace("static", "s")
df["persona"] = df["persona"].str.replace("_teacher", "")
# df = df[df["persona"] != "no"]

In [ ]:
_PERSONA_ORDERING = [
    "base", "s_short", "s_medium", "s_long",
    "d_short", "d_medium", "d_long",
    "beginner", "intermediate", "expert",
]
_order_map = {v: i for i, v in enumerate(_PERSONA_ORDERING)}

In [ ]:
def compute_gains(
    data: pd.Series,
    baseline_col: str = "helpful"
) -> pd.Series:
    """Computes the performance gain against some baseline.

    Args:
        data (pd.Series): Data used to calculate the performance gain on.
        baseline_col (str): Baseline column.

    Returns:
        pd.Series: Series with performance gains.
    """
    data = data.copy()
    for persona_ in _PERSONA_ORDERING:
        data[persona_] -= data[baseline_col]
    data = data.drop(index=baseline_col)
    return data[_PERSONA_ORDERING]

In [ ]:
tests_baseline_df = pd.read_csv("data/evaluation/tests/results_baseline.csv")
tests_baseline_model_df = pd.read_csv("data/evaluation/tests/results_baseline_model.csv")

tests_length_df = pd.read_csv("data/evaluation/tests/results_length.csv")
tests_length_model_df = pd.read_csv("data/evaluation/tests/results_length_model.csv")
tests_length_model_lr_df = pd.read_csv("data/evaluation/tests/results_length_model_lr.csv")

tests_teacher_df = pd.read_csv("data/evaluation/tests/results_teacher.csv")
tests_teacher_model_df = pd.read_csv("data/evaluation/tests/results_teacher_model.csv")
tests_teacher_model_lr_df = pd.read_csv("data/evaluation/tests/results_teacher_model_lr.csv")

tests_dynamic_df = pd.read_csv("data/evaluation/tests/results_static_vs_dynamic.csv")
tests_dynamic_model_df = pd.read_csv("data/evaluation/tests/results_static_vs_dynamic_model.csv")
tests_dynamic_model_lr_df = pd.read_csv("data/evaluation/tests/results_static_vs_dynamic_model_lr.csv")

def print_sig(
    sig_df: pd.DataFrame,
    group_col: str
):
    """Prints p-values and coefficients of significant terms.

    Args:
        sig_df (pd.DataFrame): Dataframe with significance results.
        group_col (str): Column used to group the results. Results are always grouped by dataset.
    """
    for dataset_id, dataset_subset in sig_df.groupby("dataset"):
        print(f" === {dataset_id} ===")
        for d_name, sig_group_df in dataset_subset.groupby(group_col):
            print(f"    === {d_name}")
            sig = sig_group_df[sig_group_df["pvalue"] < 0.05]
            for row in sig.itertuples():
                if row.term == "Intercept" or "llama" in row.term or "wen" in row.term or "gpt" in row.term or "gemma" in row.term\
                        or "1/2" in row.term or "no" in row.term or "0" in row.term or "1" in row.term or "2" in row.term:
                    continue
                print(f"    Persona {row.term:<20} p-val {row.pvalue:.3f} coef {row.coef:.3f}")
                if hasattr(row, "mode"):
                    print(row.mode)

# 1. Persona vs. Baseline

## Aggregate Performance Across all Datasets (except flores) and Models

In [ ]:
aggregate_df = df.copy()
dataset_df = aggregate_df[~aggregate_df["dataset"].isin(["flores", "alpaca"])]
aggregate_df = dataset_df.groupby(["persona"])["score"].mean()
aggregate_df = compute_gains(aggregate_df)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
sns.barplot(aggregate_df, orient="h", ax=ax)
plt.tight_layout()
plt.savefig("plots/results/1_0_datasets.png")

- Across all datasets, there seems to be a positive impact of dynamic personas
- Dynamic ones seem better than static
- Longer personas not necessarily better
- Teacher personas follow intuitive trend

## Performance By Dataset

In [ ]:
dataset_df = dataset_df.groupby(["dataset", "persona"], as_index=False)[["score"]].mean()

parts = []
for dataset_name, group_df in dataset_df.groupby("dataset", sort=False):
    ref = group_df.loc[group_df["persona"] == "helpful", "score"]

    ref_score = ref.iloc[0]
    group_df["reference_score"] = ref_score
    group_df["gain"] = group_df["score"] - ref_score
    parts.append(group_df)

dataset_df: pd.DataFrame = pd.concat(parts, ignore_index=True)
dataset_df = dataset_df.sort_values("persona", key=lambda s: s.map(_order_map))
dataset_df = dataset_df[dataset_df["persona"] != "helpful"]

In [ ]:
plot_kwargs = {"height": 3, "aspect": 1.2, "width": 0.8}
sns.catplot(dataset_df, col="dataset", y="persona", kind="bar", x="gain", orient="h", sharex=False,
            **plot_kwargs)
plt.savefig("plots/results/1_1_per_dataset.png")

- if we split up the comparison dataset-wise, the outcome changes a bit: it is dataset-dependent
- one some datasets, the impact is more positive: e.g. mmlu-pro and MATH
- on some it seems worse: e.g. ifbench
- there is also some variability in the performance impact per-dataset, e.g. on alpaca, dynami ones have a very large impact, on mmlu-pro not so much

In [ ]:
plotting_wrapper.create_overall_stacked_barchart("flores")

In [ ]:
plotting_wrapper.create_overall_stacked_barchart("alpaca")

In [ ]:
dataset_results = tests_baseline_df[tests_baseline_df["category"].isna()]
print_sig(dataset_results, "dataset")

- some datasets benefit more than others (e.g. mmlu-pro)
- ifbench seems completely agnostic to personas
- flores on the other hand seems very good

In [ ]:
pivot_df = df.groupby(["dataset", "persona"], as_index=False)["score"].mean()
pivot_df = pivot_df.pivot(index="persona", columns="dataset", values="score")
pivot_df = pivot_df.reindex(_order_map.keys())

fig, ax = plt.subplots(figsize=(5, 3))
sns.heatmap(data=pivot_df)
plt.tight_layout()
plt.savefig("plots/results/1_0_datasets_heatmap.png")

## Performance By Dataset Subcategory

In [ ]:
category_df = df.copy()
category_df = category_df.groupby(["dataset", "model", "category", "persona"], as_index=False)[["score"]].mean()

parts = []
for dataset_name, group_df in category_df.groupby(["category", "model"], sort=False):
    if "helpful" in group_df["persona"].unique():
        ref = group_df.loc[group_df["persona"] == "helpful", "score"]

        ref_score = ref.iloc[0]
        group_df["reference_score"] = ref_score
        group_df["gain"] = group_df["score"] - ref_score
    else:
        group_df["gain"] = group_df["score"]
    parts.append(group_df)

category_df: pd.DataFrame = pd.concat(parts, ignore_index=True)
category_df = category_df.sort_values("persona", key=lambda s: s.map(_order_map))
category_df = category_df[category_df["persona"] != "helpful"]

In [ ]:
for dataset, sub_df in category_df.groupby("dataset"):
    sns.catplot(sub_df, col="category", y="persona", kind="bar", x="gain", orient="h", sharex=False, height=3, col_wrap=3)

- similar to datasets as a whole, the impact of personas also depends on subcategories

In [ ]:
plotting_wrapper.create_stacked_barchart_plots("flores", "iso_639_3")

In [ ]:
print_sig(tests_baseline_df, "category")

- outcome also quite sensitive to categorize
- dynamic persona even stronger than before
- teachers follow mostly positive trend
- but partially, results are not significant anymore


# 2. Comparison by length

In [ ]:
from evaluate import prepare_length

dfs = []
for mode in ["static", "dynamic", "combined"]:
    mode: Literal["static", "dynamic", "combined"] = mode   # to silence warning
    length_df = prepare_length(length_base_df, mode)
    length_df["mode"] = mode
    dfs.append(length_df)
length_df: pd.DataFrame = pd.concat(dfs)

length_mapping = {1: "base", 2: "short", 3: "medium", 4: "long"}
length_df["length"] = length_df["length"].map(length_mapping)
length_df["length"] = pd.Categorical(length_df["length"], categories=length_mapping.values(), ordered=True)

In [ ]:
facet_kw = {"sharey": "row"}
sns.relplot(length_df, x="length", y="score", hue="mode", kind="line", height=3, facet_kws=facet_kw)
plt.savefig("plots/results/2_0_datasets.png")

In [ ]:
facet_kw = {"sharey": False}
sns.relplot(length_df, x="length", y="score", col="dataset", hue="mode", kind="line", facet_kws=facet_kw, height=3, col_wrap=3)
plt.savefig("plots/results/2_1_per_dataset")

In [ ]:
dataset_len_results = tests_length_df[tests_length_df["category"].isna()]
print_sig(dataset_len_results, "dataset")

- ifbench showed mostly negative trends before, so it is expected that the length will also have a negative impact
- MATH and mmlu-pro seem most intuitive, though mmlu-pro does not have a statistically significant impact
- very little strictly monotonic cases

In [ ]:
facet_kw = {"sharey": False}
d_cat_df = length_df.dropna()
for dataset, group_df in d_cat_df.groupby("dataset"):
    sns.relplot(group_df, x="length", y="score", hue="mode", col="category", kind="line", facet_kws=facet_kw, height=3, col_wrap=3)

In [ ]:
print_sig(tests_length_df, "category")

- only clear trend observable is that dynamic personas seem very strong on flores+
- also: it seems like the medium length persona is always quite bad, while longer and shorter ones are better

# 3. Comparison by expertise

In [ ]:
from evaluate import prepare_teacher

teacher_df = prepare_teacher(length_base_df)
teacher_mapping = {0: "beginner", 1: "intermediate", 2: "expert"}
teacher_df["level"] = teacher_df["level"].map(teacher_mapping)
teacher_df["level"] = pd.Categorical(teacher_df["level"], categories=["beginner", "intermediate", "expert"], ordered=True)

In [ ]:
facet_kw = {"sharey": False}
sns.relplot(teacher_df, x="level", y="score", kind="line", facet_kws=facet_kw, height=3)
plt.savefig("plots/results/3_0_datasets.png")

In [ ]:
facet_kw = {"sharey": False}
sns.relplot(teacher_df, x="level", y="score", col="dataset", kind="line", col_wrap=3, facet_kws=facet_kw, height=3)
plt.savefig("plots/results/3_1_per_dataset.png")

In [ ]:
dataset_teacher_results = tests_teacher_df[tests_teacher_df["category"].isna()]
print_sig(dataset_teacher_results, "dataset")

In [ ]:
facet_kw = {"sharey": False}
t_cat_df = teacher_df.dropna()
for dataset, group_df in t_cat_df.groupby("dataset"):
    sns.relplot(group_df, x="level", y="score", col="category", kind="line", facet_kws=facet_kw, height=3, col_wrap=3)

In [ ]:
print_sig(tests_teacher_df, "category")

# 4. Static vs. Dynamic

In [ ]:
from evaluate import prepare_static_vs_dynamic

dynamic_df = prepare_static_vs_dynamic(length_base_df)
dynamic_df.reset_index(inplace=True)

In [ ]:
sns.catplot(dynamic_df, col="dataset", x="score", y="is_dynamic", kind="point", orient="h", col_wrap=3, sharex=False, height=3, linestyle="none")
plt.savefig("plots/results/4_0_datasets.png")

In [ ]:
dataset_dynamic_results = tests_dynamic_df[tests_dynamic_df["category"].isna()]
print_sig(dataset_dynamic_results, "dataset")

In [ ]:
for dataset, dynamic_group_df in dynamic_df.groupby("dataset"):
    sns.catplot(dynamic_df, col="category", y="is_dynamic", x="score", kind="point", orient="h", col_wrap=3, sharex=False, height=2, linestyle="none")

In [ ]:
print_sig(tests_dynamic_df, "category")

# Better than nothing

In [ ]:
helpful_df = dataset_df[dataset_df["persona"].isin(["no", "helpful"])]
helpful_df = helpful_df[helpful_df["dataset"].isin(["mmlu-pro", "MATH", "ifbench"])]
sns.catplot(helpful_df, kind="bar", x="score", y="dataset", hue="persona", orient="h", aspect=2)
plt.savefig("plots/discussion/05_helpful_vs_no.png")

# Model Sensitivity

In [ ]:
# https://stats.stackexchange.com/questions/331244/how-to-test-if-an-interaction-is-significant-interaction-terms-or-model-compari

In [ ]:
def calculate_agreement(
    tests_full_df: pd.DataFrame,    # tests_length_model_df
    tests_const_df: pd.DataFrame,    # tests_length_df
    term: str
) -> pd.DataFrame:
    const_df = tests_const_df[tests_const_df["category"].isna()]
    const_df = const_df[const_df["term"] == term]
    full_df = tests_full_df[tests_full_df["term"] == term]

    res = []
    for model_group_name, model_group_df in full_df.groupby("model"):
        agreement = 0
        for data_name, data_df in model_group_df.groupby("dataset"):
            if "mode" in data_df.columns:
                for mode_name, mode_df in data_df.groupby("mode"):
                    const_sub_df = const_df[(const_df["dataset"] == data_name) & (const_df["mode"] == mode_name)]
                    sig_dataset = const_sub_df.iloc[0]["pvalue"] < 0.05
                    sig_model = mode_df.iloc[0]["pvalue"] < 0.05
                    coef_dataset = const_sub_df.iloc[0]["coef"]
                    coef_model = mode_df.iloc[0]["coef"]
                    if sig_dataset and sig_model and ((coef_dataset < 0 and coef_model < 0) or (coef_dataset > 0 and coef_model > 0)):
                        agreement += 1
                    elif not sig_dataset and not sig_model:
                        agreement += 1
            else:
                const_sub_df = const_df[const_df["dataset"] == data_name]
                sig_dataset = const_sub_df.iloc[0]["pvalue"] < 0.05
                sig_model = data_df.iloc[0]["pvalue"] < 0.05
                coef_dataset = const_sub_df.iloc[0]["coef"]
                coef_model = data_df.iloc[0]["coef"]
                if sig_dataset and sig_model and ((coef_dataset < 0 and coef_model < 0) or (coef_dataset > 0 and coef_model > 0)):
                    agreement += 1
                elif not sig_dataset and not sig_model:
                    agreement += 1
        res.append((
            model_group_name, agreement / len(const_df),
            agreement, len(const_df)
        ))
    return pd.DataFrame(
        res,
        columns=["model", "rel_agreement", "tot_agreement", "cases"]
    )

In [ ]:
whole_df = df.copy()
whole_df = whole_df.groupby(["model", "persona", "dataset"], as_index=False).mean(numeric_only=True)

parts = []
for (model_name, dataset_name), group_df in whole_df.groupby(["model", "dataset"], sort=False):
    if "helpful" in group_df["persona"].unique():
        ref = group_df.loc[group_df["persona"] == "helpful", "score"]
        ref_score = ref.iloc[0]
    else:
        ref_score = 1 - group_df["score"].mean()
    group_df["reference_score"] = ref_score
    group_df["gain"] = group_df["score"] - ref_score
    parts.append(group_df)

whole_df = pd.concat(parts)
whole_df = whole_df.sort_values("model")
whole_df = whole_df.sort_values(["model", "persona"], key=lambda s: s.map(_order_map))
whole_df = whole_df[whole_df["persona"] != "helpful"]

facet_kw = {"sharey": "row"}
sns.catplot(whole_df, x="persona", y="gain", col="model", row="dataset", kind="bar", **facet_kw)

In [ ]:
sns.catplot(whole_df, y="model", x="gain", orient="h", hue="dataset", kind="bar")
plt.savefig("plots/discussion/gain_per_model.png")

In [ ]:
whole_df.groupby(["model", "dataset"])["gain"].mean()

In [ ]:
for dataset_name, group_df in whole_df.groupby("dataset"):
    heat_df = group_df.pivot(index="persona", columns="model", values="gain")
    sns.heatmap(heat_df)
    plt.show()

In [ ]:
test = whole_df.groupby(["model", "persona"])["gain"]
test.head()

## Length

In [ ]:
tests_length_model_lr_df[tests_length_model_lr_df["pvalue"] < 0.05]

In [ ]:
length_agreement = calculate_agreement(tests_length_model_df, tests_length_df, "length")
length_agreement

In [ ]:
from collections import defaultdict

results = []
for full_test_df, const_test_df, coef_term in [
    (tests_length_model_df, tests_length_df, "length"),
    (tests_dynamic_model_df, tests_dynamic_df, "is_dynamic"),
    (tests_teacher_model_df, tests_teacher_df, "level")
]:
    tl_df = const_test_df[const_test_df["category"].isna()]
    tl_df = tl_df[tl_df["term"] == coef_term]
    tlm_df = full_test_df[full_test_df["term"] == coef_term]
    for model_name, group_df in tlm_df.groupby("model"):
        model_results = defaultdict(int)
        model_results["model"] = model_name
        model_results["type"] = coef_term
        for dataset_name, dataset_df in group_df.groupby("dataset"):
            if "mode" in dataset_df.columns:
                for mode, mode_group_df in dataset_df.groupby("mode"):
                    sub_df = tl_df[(tl_df["dataset"] == dataset_name) & (tl_df["mode"] == mode)]
                    sig_main = sub_df.iloc[0]["pvalue"] < 0.05
                    sig_mod = mode_group_df.iloc[0]["pvalue"] < 0.05
                    coef_data = sub_df.iloc[0]["coef"]
                    coef_mod = mode_group_df.iloc[0]["coef"]
                    if sig_main and sig_mod:
                        if coef_data < 0 and coef_mod < 0:
                            model_results["same"] += 1
                        elif coef_data > 0 and coef_mod > 0:
                            model_results["same"] += 1
                        elif coef_data < 0 < coef_mod:
                            model_results["neg_to_pos"] += 1
                        elif coef_data > 0 > coef_mod:
                            model_results["pos_to_neg"] += 1
                    elif sig_main:
                        if coef_data < 0 and coef_mod < 0:
                            model_results["neg_to_net"] += 1
                        elif coef_data > 0 and coef_mod > 0:
                            model_results["pos_to_net"] += 1
                        elif coef_data < 0 < coef_mod:
                            model_results["neg_to_net"] += 1
                        elif coef_data > 0 > coef_mod:
                            model_results["pos_to_net"] += 1
                    elif sig_mod:
                        if coef_data < 0 and coef_mod < 0:
                            model_results["net_to_neg"] += 1
                        elif coef_data > 0 and coef_mod > 0:
                            model_results["net_to_pos"] += 1
                        elif coef_data < 0 < coef_mod:
                            model_results["net_to_pos"] += 1
                        elif coef_data > 0 > coef_mod:
                            model_results["net_to_pos"] += 1
                    else:
                        model_results["same"] += 1
            else:
                sub_df = tl_df[tl_df["dataset"] == dataset_name]
                sig_main = sub_df.iloc[0]["pvalue"] < 0.05
                sig_mod = dataset_df.iloc[0]["pvalue"] < 0.05
                coef_data = sub_df.iloc[0]["coef"]
                coef_mod = dataset_df.iloc[0]["coef"]
                if sig_main and sig_mod:
                    if coef_data < 0 and coef_mod < 0:
                        model_results["same"] += 1
                    elif coef_data > 0 and coef_mod > 0:
                        model_results["same"] += 1
                    elif coef_data < 0 < coef_mod:
                        model_results["neg_to_pos"] += 1
                    elif coef_data > 0 > coef_mod:
                        model_results["pos_to_neg"] += 1
                elif sig_main:
                    if coef_data < 0 and coef_mod < 0:
                        model_results["neg_to_net"] += 1
                    elif coef_data > 0 and coef_mod > 0:
                        model_results["pos_to_net"] += 1
                    elif coef_data < 0 < coef_mod:
                        model_results["neg_to_net"] += 1
                    elif coef_data > 0 > coef_mod:
                        model_results["pos_to_net"] += 1
                elif sig_mod:
                    if coef_data < 0 and coef_mod < 0:
                        model_results["net_to_neg"] += 1
                    elif coef_data > 0 and coef_mod > 0:
                        model_results["net_to_pos"] += 1
                    elif coef_data < 0 < coef_mod:
                        model_results["net_to_pos"] += 1
                    elif coef_data > 0 > coef_mod:
                        model_results["net_to_pos"] += 1
                else:
                    model_results["same"] += 1
        results.append(model_results)
results_df = pd.DataFrame(results)
results_df = results_df.fillna(0)

In [ ]:
results_df = results_df.groupby("model", as_index=False).sum(numeric_only=True)
results_df

In [ ]:
sns.barplot(results_df, x="model", y="value", hue="variable")

In [ ]:
from evaluate import prepare_length

dfs = []
for mode in ["static", "dynamic", "combined"]:
    mode: Literal["static", "dynamic", "combined"] = mode  # to silence warning
    length_df = prepare_length(length_base_df, mode)
    length_df["mode"] = mode
    dfs.append(length_df)
length_df: pd.DataFrame = pd.concat(dfs)

length_mapping = {1: "base", 2: "short", 3: "medium", 4: "long"}
length_df["length"] = length_df["length"].map(length_mapping)
length_df["length"] = pd.Categorical(length_df["length"], categories=length_mapping.values(), ordered=True)

facet_kw = {"sharey": "row"}
sns.relplot(length_df, x="length", y="score", hue="model", col="mode", kind="line", height=3, facet_kws=facet_kw)
plt.savefig("plots/results/2_0_datasets.png")

## Teacher

In [ ]:
tests_teacher_model_lr_df[tests_teacher_model_lr_df["pvalue"] < 0.05]

In [ ]:
teacher_agreement = calculate_agreement(tests_teacher_model_df, tests_teacher_df, "level")
teacher_agreement

## Stat. vs. Dyn.

In [ ]:
tests_dynamic_model_lr_df[tests_dynamic_model_lr_df["pvalue"] < 0.05]

In [ ]:
dynamic_agreement = calculate_agreement(tests_dynamic_model_df, tests_dynamic_df, "is_dynamic")
dynamic_agreement

In [ ]:
total_agreement = pd.concat([length_agreement, teacher_agreement, dynamic_agreement])
for model_name, model_df in total_agreement.groupby("model"):
    print(model_name, model_df["tot_agreement"].sum() / model_df["cases"].sum())

In [ ]:
sub_df = tests_length_model_df[tests_length_model_df["term"] == "length"].round(5)
sub_df = sub_df[sub_df["dataset"] != "ifbench"]
# sub_df = sub_df.loc[sub_df["dataset"] == "mmlu-pro"]
# sub_df = sub_df.loc[sub_df["mode"] == "dynamic"]
plot_kwargs = {"sharey": False}
sns.catplot(sub_df, y="coef", col="dataset", kind="bar", x="mode", hue="model", orient="v", col_wrap=2, sharex=False, **plot_kwargs)

In [ ]:
sig_model_df = tests_length_model_df[tests_length_model_df["pvalue"] < 0.05]
sig_model_df[sig_model_df["term"]]

In [ ]:
facet_kw = {"sharey": False}
for mode in ["dynamic", "static", "combined"]:
    main_mode_df = length_df[length_df["mode"] == "dynamic"]

    g = sns.relplot(
        data=main_mode_df,
        x="length",
        y="score",
        col="dataset",
        hue="model",
        kind="line",
        errorbar=None,     # newer seaborn
        col_wrap=3,
        facet_kws=facet_kw,
    )

    g.map_dataframe(
        sns.lineplot,
        x="length",
        y="score",
        estimator="mean",
        errorbar=None,
        color="black",
        linewidth=4,
        legend=False,
    )
# sns.relplot(mode_df, x="length", y="score", col="dataset", hue="model", kind="line", err_style=None, col_wrap=3, facet_kws=facet_kw)

# Model size

In [ ]:
model_size_df = length_base_df.copy()
model_size_df = model_size_df[~model_size_df["model"].isin(["gpt-5-nano", "gpt-4-1-nano"])]
model_size_df["size"] = model_size_df["model"].isin(["qwen3-4b", "gemma-3-4b-it", "llama-3-2-3b-instruct"])
model_size_df["size"] = model_size_df["size"].replace({False: "Small", True: "Large"})

base_vis_df = model_size_df[~model_size_df["dataset"].isin(["alpaca", "flores"])]
sns.catplot(base_vis_df, col="dataset", y="persona", kind="bar", x="score", orient="h", sharex=False, hue="size")

## Length

In [ ]:
from evaluate import prepare_length

dfs = []
for mode in ["static", "dynamic", "combined"]:
    mode: Literal["static", "dynamic", "combined"] = mode   # to silence warning
    length_df = prepare_length(model_size_df, mode)
    length_df["mode"] = mode
    dfs.append(length_df)
length_df: pd.DataFrame = pd.concat(dfs)

length_mapping = {1: "base", 2: "short", 3: "medium", 4: "long"}
length_df["length"] = length_df["length"].map(length_mapping)
length_df["length"] = pd.Categorical(length_df["length"], categories=length_mapping.values(), ordered=True)

In [ ]:
facet_kw = {"sharey": False}
sns.relplot(length_df, x="length", y="score", col="dataset", hue="mode", style="size", kind="line", facet_kws=facet_kw, height=3, col_wrap=3,
            err_style=None)

In [ ]:
length_vis_df = length_df[~length_df["dataset"].isin(["ifbench", "mmlu-pro"])]

sns.relplot(length_vis_df, x="length", y="score", col="dataset", hue="mode", style="size", kind="line", facet_kws=facet_kw, height=3, col_wrap=3,
            err_style=None)
plt.savefig("plots/discussion/06_model_size_length")

## Teacher Expertise

In [ ]:
from evaluate import prepare_teacher

teacher_df = prepare_teacher(model_size_df)
teacher_mapping = {0: "beginner", 1: "intermediate", 2: "expert"}
teacher_df["level"] = teacher_df["level"].map(teacher_mapping)
teacher_df["level"] = pd.Categorical(teacher_df["level"], categories=["beginner", "intermediate", "expert"], ordered=True)

In [ ]:
facet_kw = {"sharey": False}
sns.relplot(teacher_df, x="level", y="score", col="dataset", style="size", kind="line", col_wrap=3, facet_kws=facet_kw, height=3, err_style=None)

In [ ]:
teacher_vis_df = teacher_df[teacher_df["dataset"].isin(["ifbench", "alpaca", "MATH"])]
teacher_vis_df = teacher_vis_df.sort_values(by="size")
sns.relplot(teacher_vis_df, x="level", y="score", col="dataset", style="size", kind="line", facet_kws=facet_kw, height=3, err_style=None)
plt.savefig("plots/discussion/06_model_size_teacher")

## Static vs. dynamic

In [ ]:
from evaluate import prepare_static_vs_dynamic

dynamic_df = prepare_static_vs_dynamic(model_size_df)
dynamic_df.reset_index(inplace=True)

In [ ]:
sns.catplot(dynamic_df, col="dataset", x="score", hue="is_dynamic", y="size", kind="bar", orient="h", col_wrap=3, sharex=False,
            height=3, linestyle="none")

In [ ]:
dynamic_vis_df = dynamic_df[dynamic_df["dataset"].isin(["alpaca", "flores", "mmlu-pro"])]
dynamic_vis_df = dynamic_vis_df.sort_values(by="size")
sns.catplot(dynamic_vis_df, col="dataset", x="score", hue="is_dynamic", y="size", kind="bar", orient="h", col_wrap=3, sharex=False,
            height=3, linestyle="none")
plt.savefig("plots/discussion/06_model_size_dynamic")

# Open vs. closed model

In [ ]:
length_base_df["model"].unique()

In [ ]:
oc_model_df = length_base_df.copy()
oc_model_df["Model Type"] = oc_model_df["model"].isin(["gpt-4-1-nano", "gpt-5-nano"])
oc_model_df["Model Type"] = oc_model_df["Model Type"].replace({False: "Open", True: "Closed"})

base_vis_df = oc_model_df[~oc_model_df["dataset"].isin(["alpaca", "flores"])]
sns.catplot(base_vis_df, col="dataset", y="persona", kind="bar", x="score", orient="h", sharex=False, hue="Model Type")

## Length


In [ ]:
from evaluate import prepare_length

dfs = []
for mode in ["static", "dynamic", "combined"]:
    mode: Literal["static", "dynamic", "combined"] = mode  # to silence warning
    length_df = prepare_length(oc_model_df, mode)
    length_df["mode"] = mode
    dfs.append(length_df)
length_df: pd.DataFrame = pd.concat(dfs)

length_mapping = {1: "base", 2: "short", 3: "medium", 4: "long"}
length_df["length"] = length_df["length"].map(length_mapping)
length_df["length"] = pd.Categorical(length_df["length"], categories=length_mapping.values(), ordered=True)

In [ ]:
facet_kw = {"sharey": False}
sns.relplot(length_df, x="length", y="score", col="dataset", hue="mode", style="Model Type", kind="line", facet_kws=facet_kw,
            height=3, col_wrap=3,
            err_style=None)

## Teacher Expertise


In [ ]:
from evaluate import prepare_teacher

teacher_df = prepare_teacher(oc_model_df)
teacher_mapping = {0: "beginner", 1: "intermediate", 2: "expert"}
teacher_df["level"] = teacher_df["level"].map(teacher_mapping)
teacher_df["level"] = pd.Categorical(teacher_df["level"], categories=["beginner", "intermediate", "expert"],
                                     ordered=True)
facet_kw = {"sharey": False}
sns.relplot(teacher_df, x="level", y="score", col="dataset", style="Model Type", kind="line", col_wrap=3, facet_kws=facet_kw,
            height=3, err_style=None)

## Static vs. dynamic


In [ ]:
from evaluate import prepare_static_vs_dynamic

dynamic_df = prepare_static_vs_dynamic(oc_model_df)
dynamic_df.reset_index(inplace=True)
sns.catplot(dynamic_df, col="dataset", x="score", hue="is_dynamic", y="Model Type", kind="bar", orient="h", col_wrap=3, sharex=False,
            height=3, linestyle="none")